# Online Retail: analisis agregat yang dapat diulang

Notebook ini membaca hasil build resmi, lalu mengulang EDA dan pemeriksaan KPI dengan pandas dan DuckDB SQL. Tidak ada CustomerID atau transaksi pelanggan yang ditampilkan. Jalankan build pada [README](../README.md) lebih dahulu.

Sumber: [Chen (2015), UCI Online Retail](https://doi.org/10.24432/C5BW33), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Periode resmi: 1 Desember 2010 sampai 9 Desember 2011, mata uang GBP. Desember 2011 adalah bulan parsial. Aturan lengkap ada pada [metodologi](../docs/METHODOLOGY.md).


In [1]:
from pathlib import Path
from decimal import Decimal
import json
import pandas as pd
import duckdb

candidates = [Path.cwd(), Path.cwd().parent]
if "__file__" in globals():
    candidates.append(Path(__file__).resolve().parents[1])
ROOT = next((path for path in candidates if (path / "scripts" / "build.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Buka notebook dari root repositori atau folder notebooks.")
EXPORTS = ROOT / "reports" / "exports"
names = ["monthly_country", "monthly_country_product", "customer_segments", "cohort_retention", "quality_reasons"]
required = [EXPORTS / f"{name}.csv" for name in names] + [ROOT / "reports" / "kpis.json"]
missing = [str(path.relative_to(ROOT)) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Jalankan scripts/download_data.py dan scripts/build.py lebih dahulu: {missing}")

report = json.loads((ROOT / "reports" / "kpis.json").read_text(encoding="utf-8"))
tables = {name: pd.read_csv(EXPORTS / f"{name}.csv", dtype="string") for name in names}
if report["source"].get("fixture", False):
    raise ValueError("Notebook temuan resmi tidak menerima fixture sintetis.")
print(json.dumps({key: report["source"].get(key) for key in ["name", "date_min", "date_max", "currency", "sha256"]}, indent=2))
print("\nBaris agregat:")
print(pd.Series({name: len(frame) for name, frame in tables.items()}).to_string())


{
  "name": "Online Retail",
  "date_min": "2010-12-01",
  "date_max": "2011-12-09",
  "currency": "GBP",
  "sha256": "43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d"
}

Baris agregat:
monthly_country              311
monthly_country_product    66375
customer_segments              5
cohort_retention             169
quality_reasons                5


## Kualitas data sebelum interpretasi

Deduplikasi memakai kesamaan seluruh delapan kolom sumber. Alasan/status pada CSV saling eksklusif setelah deduplikasi. Missing CustomerID adalah hitungan cakupan yang dapat tumpang tindih dengan alasan lain, sehingga jangan menambahkannya ke jumlah eksklusi. Penjualan tanpa ID yang lolos tetap masuk revenue.


In [2]:
quality = report["quality"]
quality_fields = ["rows_raw", "duplicate_rows", "rows_after_dedup", "included_sales_rows", "included_credit_rows", "excluded_rows", "missing_customer_id_rows", "unidentified_sales_rows"]
print(pd.Series({key: quality[key] for key in quality_fields}).to_string())
reasons = tables["quality_reasons"].copy()
reasons["rows"] = pd.to_numeric(reasons["rows"]).astype("int64")
assert quality["rows_raw"] == quality["duplicate_rows"] + quality["rows_after_dedup"]
assert reasons["rows"].sum() == quality["rows_after_dedup"]
assert quality["rows_after_dedup"] == quality["included_sales_rows"] + quality["included_credit_rows"] + quality["excluded_rows"]
print("\nStatus setelah deduplikasi:")
print(reasons.to_string(index=False))


rows_raw                    541909
duplicate_rows                5268
rows_after_dedup            536641
included_sales_rows         520562
included_credit_rows          8668
excluded_rows                 7411
missing_customer_id_rows    135080
unidentified_sales_rows     129412

Status setelah deduplikasi:
                      reason   rows
             included_credit   8668
               included_sale 520562
             invalid_invoice      3
        non_merchandise_code   4898
nonpositive_or_invalid_price   2510


## Tabel SQL dari ekspor publik

Kolom uang dibaca sebagai teks agar empat desimal CSV tidak berubah saat impor. SQL mengubahnya ke DECIMAL(18,4) sebelum agregasi. DataFrame hasil query di bawah mempertahankan nilai Decimal melalui fetchall(), bukan mengubahnya menjadi float.

Gross adalah penjualan positif, credits adalah absolut nilai kuantitas negatif yang lolos, dan net = gross - credits. Ini nilai merchandise tercatat, bukan profit atau matched return rate.


In [3]:
con = duckdb.connect(":memory:")
con.register("overview_csv", tables["monthly_country"])
con.register("products_csv", tables["monthly_country_product"])
con.register("cohorts_csv", tables["cohort_retention"])
con.execute("""
    CREATE VIEW overview AS
    SELECT month, country,
        CAST(gross_sales AS DECIMAL(18,4)) AS gross_sales,
        CAST(credits AS DECIMAL(18,4)) AS credits,
        CAST(net_sales AS DECIMAL(18,4)) AS net_sales,
        CAST(orders AS BIGINT) AS orders,
        CAST(units_sold AS BIGINT) AS units_sold,
        CAST(units_credited AS BIGINT) AS units_credited
    FROM overview_csv
""")
con.execute("""
    CREATE VIEW products AS
    SELECT month, country, stock_code, description,
        CAST(gross_sales AS DECIMAL(18,4)) AS gross_sales,
        CAST(credits AS DECIMAL(18,4)) AS credits,
        CAST(net_sales AS DECIMAL(18,4)) AS net_sales
    FROM products_csv
""")

def query_frame(sql):
    result = con.execute(sql)
    return pd.DataFrame(result.fetchall(), columns=[column[0] for column in result.description])

totals = query_frame("""
    SELECT SUM(gross_sales) AS gross_sales, SUM(credits) AS credits,
        SUM(net_sales) AS net_sales, SUM(orders) AS orders,
        SUM(units_sold) AS units_sold, SUM(units_credited) AS units_credited
    FROM overview
""")
for field in totals.columns:
    assert Decimal(str(totals.iloc[0][field])) == Decimal(str(report["kpis"][field])), field
assert totals.iloc[0]["gross_sales"] - totals.iloc[0]["credits"] == totals.iloc[0]["net_sales"]
product_net = con.execute("SELECT SUM(net_sales) FROM products").fetchone()[0]
assert product_net == totals.iloc[0]["net_sales"]
print(totals.to_string(index=False))
gross = totals.iloc[0]["gross_sales"]
credits = totals.iloc[0]["credits"]
orders = int(totals.iloc[0]["orders"])
print(f"Gross AOV: GBP {gross / Decimal(orders):,.2f}")
print(f"Credit value rate: {credits / gross:.2%}")


  gross_sales     credits    net_sales  orders  units_sold  units_credited
10216121.2700 475901.1600 9740220.1100   19771     5556332          270017
Gross AOV: GBP 516.72
Credit value rate: 4.66%


## Tren bulanan dengan periode yang setara

Kelengkapan bulan mengikuti cakupan kalender sumber. Desember 2010 penuh, sedangkan Desember 2011 hanya sampai tanggal 9. Urutkan bulan penuh untuk membandingkan puncak penjualan, lalu tampilkan bulan parsial secara terpisah. Perubahan antarbulan di sini deskriptif dan tidak membuktikan dampak pemasaran.


In [4]:
monthly = query_frame("""
    SELECT month, SUM(gross_sales) AS gross_sales, SUM(credits) AS credits,
        SUM(net_sales) AS net_sales, SUM(orders) AS orders
    FROM overview GROUP BY month ORDER BY month
""")
first = pd.Timestamp(report["source"]["date_min"])
last = pd.Timestamp(report["source"]["date_max"])
month_start = pd.to_datetime(monthly["month"] + "-01")
month_end = month_start + pd.offsets.MonthEnd(0)
monthly["complete_month"] = (month_start >= first) & (month_end <= last)
print(monthly.to_string(index=False))
complete_months = monthly.loc[monthly["complete_month"], "month"].tolist()
peak = monthly.loc[monthly["complete_month"]].sort_values("net_sales", ascending=False).head(1)
print("\nPuncak bulan penuh:")
print(peak[["month", "net_sales"]].to_string(index=False))
assert "2010-12" in complete_months
assert "2011-12" not in complete_months


  month  gross_sales     credits    net_sales  orders  complete_month
2010-12  769401.2100  17547.8600  751853.3500    1549            True
2011-01  669366.5700  91525.0500  577841.5200    1081            True
2011-02  505765.7300   8335.5200  497430.2100    1093            True
2011-03  687543.4800  10504.3000  677039.1800    1440            True
2011-04  513574.4100  33316.0600  480258.3500    1235            True
2011-05  737250.0700   8948.2300  728301.8400    1668            True
2011-06  735675.0900  13713.8400  721961.2500    1525            True
2011-07  686815.9900  11407.2600  675408.7300    1451            True
2011-08  722516.6200  22926.5700  699590.0500    1339            True
2011-09 1026362.7500  16980.6400 1009382.1100    1818            True
2011-10 1100707.9200  41613.0300 1059094.8900    2005            True
2011-11 1448180.2400  24991.4400 1423188.8000    2751            True
2011-12  612961.1900 174091.3600  438869.8300     816           False

Puncak bulan penuh:

## Konsentrasi negara

Country adalah tempat tinggal pelanggan menurut sumber. Share revenue membantu menilai konsentrasi historis. Share tersebut tidak menunjukkan ukuran pasar, permintaan yang belum terlayani, atau tempat kirim.


In [5]:
country = query_frame("""
    SELECT country, SUM(gross_sales) AS gross_sales,
        SUM(credits) AS credits, SUM(net_sales) AS net_sales, SUM(orders) AS orders
    FROM overview GROUP BY country ORDER BY net_sales DESC, country
""")
country["net_share"] = country["net_sales"].map(lambda value: value / totals.iloc[0]["net_sales"])
print(country.head(10).to_string(index=False, formatters={"net_share": lambda value: f"{value:.2%}"}))
assert sum(country["net_sales"], Decimal("0")) == totals.iloc[0]["net_sales"]


       country  gross_sales     credits    net_sales  orders net_share
United Kingdom 8693345.6200 443053.3100 8250292.3100   17899    84.70%
   Netherlands  283889.3400    409.8000  283479.5400      93     2.91%
          EIRE  270850.8600  11470.8400  259380.0200     282     2.66%
       Germany  205381.1500   4761.4900  200619.6600     443     2.06%
        France  184493.0000   2416.4000  182076.6000     382     1.87%
     Australia  138103.8100   1181.3100  136922.5000      56     1.41%
   Switzerland   53065.6000    582.5500   52483.0500      50     0.54%
         Spain   55706.5600   3959.9100   51746.6500      88     0.53%
       Belgium   36927.3400    264.3800   36662.9600      98     0.38%
         Japan   37416.3700   1996.5800   35419.7900      19     0.36%


## Produk dan kredit

Ranking berikut menjumlahkan nilai uang per SKU. Jumlah pesanan per SKU sengaja tidak dijumlahkan lintas produk: invoice yang sama dapat membeli beberapa SKU. Contoh kedua memakai batas gross GBP 1.000 agar rasio tinggi dari basis yang kecil tidak langsung mendominasi daftar. Batas ini pilihan eksplorasi, bukan ambang keputusan bisnis.


In [6]:
top_products = query_frame("""
    SELECT stock_code, description, SUM(gross_sales) AS gross_sales,
        SUM(credits) AS credits, SUM(net_sales) AS net_sales
    FROM products
    GROUP BY stock_code, description
    ORDER BY net_sales DESC, stock_code LIMIT 10
""")
print("Sepuluh SKU dengan net sales tertinggi:")
print(top_products.to_string(index=False))
credit_review = query_frame("""
    SELECT stock_code, description, SUM(gross_sales) AS gross_sales,
        SUM(credits) AS credits, SUM(net_sales) AS net_sales,
        SUM(credits) / NULLIF(SUM(gross_sales), 0) AS credit_value_rate
    FROM products
    GROUP BY stock_code, description
    HAVING SUM(gross_sales) >= 1000 AND SUM(credits) > 0
    ORDER BY credit_value_rate DESC, stock_code LIMIT 10
""")
print("\nContoh daftar untuk peninjauan kredit (gross >= GBP 1.000):")
print(credit_review.to_string(index=False, formatters={"credit_value_rate": lambda value: f"{value:.2%}"}))


Sepuluh SKU dengan net sales tertinggi:
stock_code                        description gross_sales   credits   net_sales
     22423           REGENCY CAKESTAND 3 TIER 174156.5400 9697.0500 164459.4900
     47566                      PARTY BUNTING  99445.2300 1201.3500  98243.8800
    85123A WHITE HANGING HEART T-LIGHT HOLDER 104462.7500 6624.3000  97838.4500
    85099B            JUMBO BAG RED RETROSPOT  94159.8100 1984.0200  92175.7900
     23084                 RABBIT NIGHT LIGHT  66870.0300  208.4000  66661.6300
     22086     PAPER CHAIN KIT 50'S CHRISTMAS  64875.5900 1160.3500  63715.2400
     84879      ASSORTED COLOUR BIRD ORNAMENT  58927.6200  135.2000  58792.4200
     79321                      CHILLI LIGHTS  54096.3600  349.7000  53746.6600
     22502         PICNIC BASKET WICKER SMALL  51408.7700  385.2500  51023.5200
     22197                     POPCORN HOLDER  51334.4700  366.5500  50967.9200

Contoh daftar untuk peninjauan kredit (gross >= GBP 1.000):
stock_code         

## RFM dan kelompok R/F global

R, F, dan M dihitung pada penjualan positif yang memiliki CustomerID. Monetary adalah gross spend sebelum kredit. Keanggotaan segmen memakai R/F dengan urutan aturan di SQL, tanpa ambang Monetary atau skor kuintil. Karena ekspor hanya agregat segmen, notebook ini tidak menghitung ulang perilaku per pelanggan.


In [7]:
segments = tables["customer_segments"].copy()
segments["customers"] = pd.to_numeric(segments["customers"]).astype("int64")
segments["monetary"] = segments["monetary"].map(Decimal)
customers = report["customers"]
assert segments["customers"].sum() == customers["identified_buyers"]
repeat_from_segments = segments.loc[segments["segment"] != "One-time buyers", "customers"].sum()
assert int(repeat_from_segments) == customers["repeat_buyers"]
print(f"Tanggal acuan: {customers['as_of']}")
print(segments.to_string(index=False))
print(f"Repeat buyers: {customers['repeat_buyers']:,} / {customers['identified_buyers']:,} ({customers['repeat_rate']:.2%})")
print("Cakupan: identified positive-sales buyers, seluruh periode, tanpa filter negara/bulan.")


Tanggal acuan: 2011-12-10
               segment  customers     monetary avg_recency_days avg_frequency
       One-time buyers       1505  626605.9000         156.9734           1.0
   Other repeat buyers       1204 1421534.1200          34.6304        2.8439
Recent frequent buyers        780 5256581.3700          11.6218       12.5436
  Lapsed repeat buyers        595  649020.4600         175.4202        3.0454
 Other frequent buyers        250  783485.7900           50.656         7.508
Repeat buyers: 2,829 / 4,334 (65.27%)
Cakupan: identified positive-sales buyers, seluruh periode, tanpa filter negara/bulan.


## Retention cohort dan bulan yang belum teramati

Cohort memakai bulan pembelian pertama yang terlihat dalam sumber. Bulan 0 harus 100%. Sel masa depan kosong; 0% hanya untuk periode yang sudah teramati tanpa pembeli aktif. Aktivitas Desember 2011 tetap parsial. Tabel tampilan memuat periode 0 sampai 6 agar mudah dibaca; objek pivot menyimpan seluruh periode.


In [8]:
cohorts = tables["cohort_retention"].copy()
for field in ["period_index", "customers", "cohort_size"]:
    cohorts[field] = pd.to_numeric(cohorts[field]).astype("Int64")
cohorts["retention_rate"] = pd.to_numeric(cohorts["retention_rate"]).astype("Float64")
cohorts["observed"] = cohorts["observed"].str.lower().eq("true")
month_zero = cohorts.loc[cohorts["period_index"] == 0]
assert (month_zero["customers"] == month_zero["cohort_size"]).all()
assert month_zero["retention_rate"].eq(1).all()
future = cohorts.loc[~cohorts["observed"]]
assert future["customers"].isna().all()
assert future["retention_rate"].isna().all()
pivot = cohorts.pivot(index="cohort_month", columns="period_index", values="retention_rate")
print(pivot.loc[:, pivot.columns <= 6].to_string(na_rep="not observed", float_format=lambda value: f"{value:.1%}"))
print(f"Sel future yang dibiarkan kosong: {len(future)}")


period_index      0     1     2     3     4     5     6
cohort_month                                           
2010-12      100.0% 36.5% 32.4% 38.3% 36.2% 39.8% 36.2%
2011-01      100.0% 21.9% 26.7% 22.8% 31.7% 28.8% 24.8%
2011-02      100.0% 18.7% 18.7% 28.7% 27.1% 24.5% 25.5%
2011-03      100.0% 14.8% 25.2% 19.9% 22.3% 16.8% 26.8%
2011-04      100.0% 21.0% 20.3% 21.0% 19.7% 22.7% 21.7%
2011-05      100.0% 19.0% 17.3% 17.3% 20.8% 23.2% 26.4%
2011-06      100.0% 17.4% 15.7% 26.4% 23.1% 33.1%  9.5%
2011-07      100.0% 17.6% 20.3% 22.5% 27.3% 11.2%  <NA>
2011-08      100.0% 20.1% 24.3% 24.3% 12.4%  <NA>  <NA>
2011-09      100.0% 23.4% 30.1% 11.4%  <NA>  <NA>  <NA>
2011-10      100.0% 23.5% 11.5%  <NA>  <NA>  <NA>  <NA>
2011-11      100.0% 11.1%  <NA>  <NA>  <NA>  <NA>  <NA>
2011-12      100.0%  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>
Sel future yang dibiarkan kosong: 78


## Ringkasan retention dengan denominator yang benar

Untuk periode 1, setiap cohort menyumbang tepat satu denominator. Hanya activity_month yang merupakan bulan penuh disertakan. Penjumlahan pembeli aktif dibagi jumlah ukuran cohort menghasilkan rata-rata berbobot. Mengambil rata-rata sederhana dari persentase sel akan memberi bobot sama pada cohort besar dan kecil.


In [9]:
period_one = cohorts.loc[
    cohorts["period_index"].eq(1)
    & cohorts["observed"]
    & cohorts["activity_month"].isin(complete_months)
].copy()
active = int(period_one["customers"].sum())
base = int(period_one["cohort_size"].sum())
weighted = Decimal(active) / Decimal(base) if base else None
print(period_one[["cohort_month", "activity_month", "customers", "cohort_size", "retention_rate"]].to_string(index=False))
print(f"Periode 1 pada bulan aktivitas penuh: {active:,} / {base:,} = {weighted:.2%}" if weighted is not None else "Belum ada periode 1 yang sebanding.")


cohort_month activity_month  customers  cohort_size  retention_rate
     2010-12        2011-01        323          884        0.365385
     2011-01        2011-02         91          416         0.21875
     2011-02        2011-03         71          380        0.186842
     2011-03        2011-04         67          452         0.14823
     2011-04        2011-05         63          300            0.21
     2011-05        2011-06         54          284        0.190141
     2011-06        2011-07         42          242        0.173554
     2011-07        2011-08         33          187        0.176471
     2011-08        2011-09         34          169        0.201183
     2011-09        2011-10         70          299        0.234114
     2011-10        2011-11         84          357        0.235294
Periode 1 pada bulan aktivitas penuh: 932 / 3,970 = 23.48%


## Temuan hasil build dan tindak lanjut

Teks berikut berasal dari query pipeline pada build yang sedang dibaca. Setiap temuan menyertakan cakupan. Gunakan daftar ini sebagai awal presentasi, lalu nyatakan data tambahan yang diperlukan sebelum tindakan: biaya untuk profit, alasan kredit untuk investigasi, dan kelompok kontrol untuk kampanye.

Pertanyaan latihan: apakah konsentrasi revenue memerlukan mitigasi risiko? Apakah SKU dengan nilai kredit besar perlu pemeriksaan manual? Bagaimana menguji win-back untuk kelompok lapsed tanpa menyatakan korelasi sebagai efek kampanye?


In [10]:
for finding in report["findings"]:
    print(f"{finding['title']}\n{finding['detail']}\nScope: {finding['scope']}\n")
print("Pemeriksaan agregat notebook: PASS")
con.close()


Merchandise credits need separate monitoring
Credits equal 4.66% of gross merchandise sales. This is a credit-value ratio, not a matched physical-return rate.
Scope: full_period

Repeat purchase among identified buyers
2,829 of 4,334 identified buyers have at least two positive invoices (65.27%). Anonymous purchases are outside this denominator.
Scope: full_period_identified_sales_buyers

Unidentified sales remain in the financial analysis
GBP 1,478,893.6300 in gross merchandise sales lacks a customer ID; it remains in revenue but cannot be assigned to customer segments or cohorts.
Scope: full_period

Peak net-sales month among complete months
2011-11 records GBP 1,423,188.8000 net merchandise sales. Partial source months are excluded from this comparison.
Scope: complete_months

Pemeriksaan agregat notebook: PASS
